# Free-Droid (Szabi) — v11 fine-tune (új dataset + epoch 3, epochonkénti checkpointtal)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

## Mi ez a kör

Két dolog változik a v10-hez képest, és **ez látszólag megsérti az egyváltozós elvet** — ezért
itt az indoklás:

1. **Új dataset (915 → 976).** A v10 három regressziójának adatoldali javítása: 18 hosszú (100–106
   szavas) kifejtős példa + 8 rövid kontraszt, 35 köszönés, és a megszólítás egységesítve
   („Teremtőm") + 24% → 50% arányra emelve.
2. **`epochs` 1 → 3**, a `--preset gentle` többi értéke (`lr=5e-5`, `r=8`, `alpha=8`) változatlan.

### Miért nem két külön kör

Mert 1 epochnál a dataset-javítás **nem mérhető**: a 18 új hosszú példa fejenként pontosan egyszer
kerülne a modell elé, r=8-as adapteren, negyedelt tanulási rátán. Ha így marad a koherencia 0/3, nem
lehet megkülönböztetni, hogy „a dataset nem működik" vagy „1 epoch nem elég hozzá". Az egyváltozós
elv csak akkor véd, ha a kontrollkar érvényes rezsimben van — itt nem az.

### És miért nem baj, hogy két változó mozog

Mert **külön műszerük van**:

| Változó | Műszer |
| :-- | :-- |
| epoch-szám | a loss-görbe (train vs eval divergencia) — benchmark nélkül |
| dataset | `compare_epochs.py`: válaszhossz, köszönés-viszonzás, megszólítás-arány |

És a döntő: **több epoch egy tömör korpuszon rövidebb választ ad, nem hosszabbat.** Ha hosszú,
összeálló válaszok jelennek meg, azt csak az adat okozhatta.

### Amit ez a kör NEM változtat

**`lora_r` marad 8.** Az r=8→16 csábító (a `gentle` preset a hibás, maszkolás nélküli rezsimben
született, tehát az indoka gyanús), de az **v12** — annak nincs önálló műszere, saját futás kell hozzá.

## Az alultanulás bizonyítéka, amiért az epoch emelkedik

A v10-ben **`eval_loss < train_loss` MINDKÉT méreten** (3B 2,118 < 2,205; 8B 1,635 < 1,713), a görbe
az utolsó lépésnél is ereszkedett, és összesen **1 epoch / 103 lépés** futott.

## A baseline, amit verni kell (`compare_epochs.py` a v10 raw-ján)

| | v10 8B | v10 3B | v11 cél |
| :-- | --: | --: | --: |
| koherencia-válasz átlaghossz | **25 szó** | 55 szó | ~100 |
| köszönést viszonoz | **0/2** | 2/2 | 2/2 |
| megszólítás („Teremtőm") | **0%** | 4% | 50% felé |

A 8B tehát a koherencia-kérdéseken **rövidebb, mint a 3B**, és egyszer sem viszonozza a köszönést.


## Unsloth telepítése

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes


## Repo klónozása + guard

In [ ]:
# 2. Repo a kívánt ágról, majd be a training/-be.
# PR-staging alatt állítsd a feature-ágra; merge után hagyd "main"-en.
import os, shutil

BRANCH = "main"

# Újrafuttatható: előbb vissza /content-be és el a korábbi klónnal. Enélkül a cella
# második futtatása a training/ ALÁ klónozna (a %cd megmarad a session-ben).
%cd /content
if os.path.exists("free-droid"):
    shutil.rmtree("free-droid")
!git clone --depth 1 -b {BRANCH} https://github.com/pits2022/free-droid.git
%cd free-droid/training

# A guard a repóban él (training/verify_training_setup.py) — a v11 LÉNYEGÉT is nézi:
# 976 példa ÉS legalább 15 db 100+ szavas példa. Ez utóbbi az, amin ez a kör áll vagy
# bukik: ha a hosszú példák nincsenek ott, a futás értelmetlen, és a hiba NEM az
# epoch-számban lenne keresendő. Régi ágról klónozva ez azonnal kiderül, nem 1,5 óra múlva.
#
# subprocess + assert, NEM `!python`: a `!parancs` nem-nulla kilépési kódja Jupyterben
# nem állítja meg a notebookot, tehát a guard csak figyelmeztetne. Az assert viszont
# elhasal, és a "Run all" is megáll rajta.
import subprocess, sys

rc = subprocess.run([sys.executable, "verify_training_setup.py"]).returncode
assert rc == 0, "A pre-flight guard BUKOTT — ne indítsd a tanítást (lásd a kimenetet fent)."


# === A KIMENET A KLÓNON KÍVÜL ÉL — ez nem kényelmi kérdés ===
# A fenti `shutil.rmtree("free-droid")` a cella ÚJRAFUTTATÁSAKOR (pl. ágváltásnál) az
# egész könyvtárat törli. Ha a training/outputs/ a klónon BELÜL van, ezzel megy a több
# órányi tanítás eredménye is — checkpointok, adapter, GGUF. Egyszer meg is történt.
# Ezért az outputs egy szimlink egy klónon kívüli mappára.
import pathlib

KIMENET = pathlib.Path("/content/outputs")   # 8B-nél tedd Drive-ra, lásd a 8B cella jegyzetét
KIMENET.mkdir(parents=True, exist_ok=True)

_link = pathlib.Path("outputs")
if _link.is_symlink():
    _link.unlink()
elif _link.exists():
    shutil.rmtree(_link)
os.symlink(KIMENET, _link)
print(f"outputs -> {KIMENET}  (a klón törlése ezt már nem érinti)")
!ls -A {KIMENET} 2>/dev/null || true

!wc -l dataset/train.jsonl dataset/val.jsonl


## Edge modell — Llama 3.2 3B (offline fallback)

A **rövidített** `system_prompt_3b.txt`-vel tanul. Figyeld a `response masking:` sort — a rövidebb
prompt miatt itt ~6% körüli aránynak kell jönnie.

⚠️ **Figyeld a `eval_loss` sorokat epochonként.** Ez a VÉTÓ: ha az eval_loss emelkedni kezd, az az
epoch (és minden utána) túltanult, és kiesik a jelöltek közül. Ha végig ereszkedik, egyik sem esik ki
— és akkor a loss ennél többet nem mond, a választás a `compare_epochs.py`-ra megy át.

In [ ]:
!python finetune.py --variant llama --preset gentle --epochs 3 --tag v11

## Cloud modell — Llama 3.1 8B (a fő demó-agy, CPU-cloud)

A **kanonikus** `system_prompt.txt`-vel tanul, ezért a `response masking:` arány itt alacsonyabb,
~4% körüli — ugyanaz a válasz, hosszabb prompt mellett.

⏱️ **IDŐ-KOCKÁZAT — olvasd el, mielőtt elindítod.** A 3B ugyanezen a 330 lépésen **1 óra 23 percet**
futott (15,2 s/lépés, evallal együtt), plusz ~20 perc export. A 8B lépésenként nagyságrendileg
2,5–3× lassabb, tehát **~3,5–4 óra tanítás + ~40 perc export**. Ez több, mint amennyit egy ingyenes
Colab-session általában kibír.

**A checkpoint-mentés itt csak akkor ér valamit, ha túléli a szakadást.** A `/content` a runtime-mal
együtt elveszik — az epochonkénti adapter is. Ha nem akarod újrakezdeni, a tanítás ELŐTT tedd a
kimenetet Drive-ra:

```python
from google.colab import drive; drive.mount('/content/drive')
import pathlib, os, shutil
KIMENET = pathlib.Path('/content/drive/MyDrive/free-droid-outputs')
KIMENET.mkdir(parents=True, exist_ok=True)
_link = pathlib.Path('outputs')
if _link.is_symlink(): _link.unlink()
elif _link.exists(): shutil.copytree(_link, KIMENET, dirs_exist_ok=True); shutil.rmtree(_link)
os.symlink(KIMENET, _link)
print(f'outputs -> {KIMENET}')
```

Enélkül is elindulhat — de akkor **egy szakadás az egész 4 órát viszi**, nem csak az utolsó epochot.


In [ ]:
!python finetune.py --variant llama8b --preset gentle --epochs 3 --tag v11

## Epoch-jelöltek exportálása (3B)

A `finetune.py` a tanítás VÉGI modelljét exportálja — az tehát **az epoch 3** (`gguf-q4_k_m`).
Az epoch 1 és 2 a `checkpoints/` alatt van mentve; azokat az `export_checkpoint.py` teszi
futtathatóvá.

**Csak a 3B-t exportáljuk itt mindhárom epochra**, mert ott dől el az epoch-szám olcsón:
a 3B export percek, a **8B exportonként ~15–25 perc és több tíz GB átmeneti hely**. A 8B-nél
majd csak a nyertes epochot (és szükség esetén a szomszédját) exportáld — az utolsó cellában
készen van hozzá a parancs.

In [ ]:
# 3. Az epoch 1 és 2 exportálása a 3B-ből (az epoch 3 már kész: gguf-q4_k_m_gguf).
import shutil, subprocess, sys

!python export_checkpoint.py --variant llama --tag v11 --list

for epoch in (1, 2):
    # Szabad hely ELLENŐRZÉSE exportonként: az Unsloth 16-bitre olvaszt (~6 GB), abból
    # csinál F16 GGUF-ot (~6 GB), majd kvantál (~2 GB). Ha elfogy a lemez, a hiba mélyen
    # az Unsloth belsejéből jön, és semmi köze a checkpointhoz — ne ott keresd.
    szabad = shutil.disk_usage(".").free / 2**30
    print(f"\n=== epoch {epoch} export — szabad hely: {szabad:.1f} GB ===")
    if szabad < 20:
        print("  ⚠️  20 GB alatt az export elhasalhat. Törölhető: a korábbi 16-bites merge")
        print("      mappák (gguf-q4_k_m/, gguf-*-e*/), a GGUF-ok a _gguf mappákban vannak.")

    # capture_output=False, hogy élőben lásd a haladást; a végén a returncode számít.
    rc = subprocess.run([sys.executable, "export_checkpoint.py",
                         "--variant", "llama", "--tag", "v11",
                         "--epoch", str(epoch), "--quants", "q4_k_m"]).returncode
    assert rc == 0, (
        f"az epoch {epoch} exportja bukott (exit {rc}). A VALÓDI hibaüzenet ENNEK A "
        f"CELLÁNAK A KIMENETÉBEN van, feljebb — az assert csak megállítja a Run all-t. "
        f"Külön is futtatható: !python export_checkpoint.py --variant llama --tag v11 "
        f"--epoch {epoch} --quants q4_k_m")

!ls -d outputs/llama3.2-3b-v11/gguf-* outputs/llama3.2-3b-v11/lora-adapter*
!df -h /content | tail -1


## Next

- **Kimenetek:** `training/outputs/llama3.2-3b-v11/` és `.../llama3.1-8b-v11/`.
  ⚠️ **A `lora-adapter*` mappákat IS töltsd le**, ne csak a GGUF-ot. A v8-nál csak a GGUF került le,
  az adapter a Colab-runtime-mal együtt majdnem elveszett — a HF Space viszont az ADAPTERT tölti be,
  GGUF-fal nem lehet átállítani. A `checkpoints/` mappa is jöjjön le, ha később másik epoch kell.

- ⚠️ **AZ UNSLOTH ÁLTAL GENERÁLT `Modelfile`-t NE HASZNÁLD.** A futás vége ezt ajánlja
  (`ollama create model_name -f .../Modelfile`), de abban **nincs SYSTEM**, és **temperature 1.5**
  van benne (llama.cpp export-default). SYSTEM nélkül a modell más kontextust kap, mint amin
  tanult — csendben romlik a viselkedés —, a temperature 1.5 szórása pedig elmossa a mérésben
  azt a különbséget, amit épp keresünk. Mindig a `make_modelfile.py` kell, az a `config.py`-ból
  veszi a variánshoz tartozó promptot (a 3B a rövidített `system_prompt_3b.txt`-t).

- ⚠️ **A GGUF nem ott van, ahol az export-mappa neve mutatja.** Az Unsloth a megadott mappába a
  **16-bites merge-et** teszi, a GGUF-ot pedig egy `_gguf` utótagú testvérmappába:
  ```
  outputs/llama3.2-3b-v11/gguf-q4_k_m/          <- 16-bit merge (NEM ez kell)
  outputs/llama3.2-3b-v11/gguf-q4_k_m_gguf/     <- itt a .Q4_K_M.gguf
  ```
  Így a parancs (az `-e1` az export_checkpoint.py-ból jövő epoch-jelölteknél):
  ```
  python make_modelfile.py --variant llama \
      outputs/llama3.2-3b-v11/gguf-q4_k_m-e1_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf
  cd outputs/llama3.2-3b-v11/gguf-q4_k_m-e1_gguf && ollama create szabi-3b-v11e1 -f Modelfile_<név>
  ```

### 1. lépés — mechanikus epoch-választás (ez dönt)

```
python run_benchmark.py --models szabi-3b-v11e1 szabi-3b-v11e2 szabi-3b-v11e3 --json-out --no-blind
python compare_epochs.py benchmark_raw_<dátum>.json
```

A `--no-blind` itt szándékos: ez **mechanikus mérés**, nem emberi pontozás, tehát a vakítás csak
akadályozna. A táblát a fenti baseline-hoz mérd. Amit nézel:

1. **koherencia-válaszhossz** — ez a kör tétje. Ha egy oszlop a v10 szintjén (25–55 szó) marad, az a
   checkpoint nem tanulta meg az új adatot.
2. **köszönést viszonoz** — 2/2 a cél.
3. **kitalált tool-név / csupasz tool-hívás** — romlás-őrök. Ha nőnek, a hosszabb tanítás mást
   rontott el, és **a kisebb epoch-szám nyer**.

### 2. lépés — a 8B exportja a nyertes epochra

```
python export_checkpoint.py --variant llama8b --tag v11 --epoch <nyertes> --quants q4_k_m
```

### 3. lépés — vak, bináris aréna (a „vállalható-e a demóra" kérdés)

```
python -m freedroid.rag.corpus       # MINDIG, mielőtt --rag-gal mérsz
python run_benchmark.py --models szabi-8b-v11 szabi-3b-v11 --rag \
    --anchor benchmark_raw_2026-07-29_v10.json
# kézi pontozás a generált .md-ben (0/1 + ok-címke), majd:
python run_benchmark.py --decode benchmark_eredmeny_<dátum>.md --key benchmark_kulcs_<dátum>.json
```

A horgony a v10-ből jön: ha ugyanazok a válaszok ma más pontot kapnak, az a **pontozó** driftje, nem
a modellé. Korlát, amit tudni kell: n=25-nél a bináris arány CI-je 45–83% — „kész-e a demóra"-ra jó,
„jobb-e 5%-kal"-ra nem.

### 4. lépés — red team (a demó előtt kötelező)

```
python run_benchmark.py --models szabi-8b-v11 szabi-3b-v11 --benchmark-file red_team.json \
    --rag --rag-dims halluc_absztencio
```

### Amit ez a kör NEM old meg

- A benchmark **8 kérdése szó szerint benne van a tanítóadatban** (memorizációt mér) — ezért fut a
  szivárgás-ellenőrzés `--baseline`-nal.
- 24 scaffold-szennyezett példa (index 476–599).
- `lora_r` 8 → 16: **v12**, saját futással.
